## ML Gymnasium Environment

In [6]:
%pip install gymnasium


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Chargement du tenseur

In [10]:
import torch

initial_states_tensor = torch.load("initial_states.pt")
print(f"Tenseur chargé. Shape : {initial_states_tensor.shape}")

Tenseur chargé. Shape : torch.Size([42, 6])


In [8]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class ATCEnv(gym.Env):
    def __init__(self, initial_states_tensor):
        super(ATCEnv, self).__init__()
        
        # On repasse en NumPy pour les calculs physiques internes de l'environnement
        self.initial_states = initial_states_tensor.numpy() 
        self.current_step = 0
        self.max_steps = 60  # Durée max d'un épisode (ex: 60 itérations)
        
        # === ACTION SPACE (Espace discret des manœuvres ATC) ===
        # 0 : Maintenir la trajectoire
        # 1 : Virage à droite (+15°)
        # 2 : Virage à gauche (-15°)
        # 3 : Monter (+1000 ft)
        # 4 : Descendre (-1000 ft)
        self.action_space = spaces.Discrete(5)
        
        # === OBSERVATION SPACE ===
        # Vecteur de 6 dimensions : [dx, dy, dz, dvx, dvy, dvz] normalisés
        # On met des limites larges (-10 à +10 écarts-types)
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(6,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        """Initialise un nouvel épisode en piochant un conflit au hasard."""
        super().reset(seed=seed)
        
        # Sélection aléatoire d'un des 42 scénarios
        idx = np.random.randint(len(self.initial_states))
        self.state = self.initial_states[idx].copy()
        
        self.current_step = 0
        
        return self.state, {}

    def step(self, action):
        """Exécute la manœuvre et calcule le nouvel état physique."""
        # 1. Mise à jour de la cinématique (vitesses puis positions) selon l'action
        # TODO: Moteur physique à coder
        
        # 2. Vérification des distances (Séparation ATC violée ou conflit résolu ?)
        # TODO: Calcul de la géométrie post-action
        
        # 3. Attribution de la récompense
        # TODO: Fonction de coût (Reward)
        
        self.current_step += 1
        
        # Conditions d'arrêt : collision, sortie de zone, ou temps écoulé
        terminated = False  # Passe à True si crash ou réussite totale
        truncated = self.current_step >= self.max_steps
        reward = 0.0
        
        return self.state, reward, terminated, truncated, {}

# Test rapide de l'instanciation
env = ATCEnv(initial_states_tensor)
obs, info = env.reset()
print(f"Environnement initialisé. Première observation tirée au sort :\n{obs}")

Environnement initialisé. Première observation tirée au sort :
[ 0.84486306 -0.02280809 -1.3068094  -0.22692251  0.41464686  0.8080247 ]


In [12]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class ATCEnv(gym.Env):
    def __init__(self, initial_states_tensor):
        super(ATCEnv, self).__init__()
        
        self.initial_states = initial_states_tensor.numpy()
        self.current_step = 0
        self.max_steps = 50  
        self.dt = 10.0       
        
        self.action_space = spaces.Discrete(5)
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(6,), dtype=np.float32
        )
        self.prev_dist = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = np.random.randint(len(self.initial_states))
        self.state = self.initial_states[idx].copy()
        self.current_step = 0
        
        # On mémorise la distance horizontale initiale pour calculer le progrès
        dx, dy, _, _, _, _ = self.state
        self.prev_dist = np.hypot(dx, dy)
        
        return self.state, {}

    def step(self, action):
        dx, dy, dz, dvx, dvy, dvz = self.state
        
        turn_rate = 0.05
        climb_rate = 0.1
        
        if action == 1:   # Virage à droite
            dvy += turn_rate
        elif action == 2: # Virage à gauche
            dvy -= turn_rate
        elif action == 3: # Montée
            dz += climb_rate
            dvz += climb_rate
        elif action == 4: # Descente
            dz -= climb_rate
            dvz -= climb_rate
            
        dx += dvx * (self.dt / 60.0)
        dy += dvy * (self.dt / 60.0)
        dz += dvz * (self.dt / 60.0)
        
        self.state = np.array([dx, dy, dz, dvx, dvy, dvz], dtype=np.float32)
        
        curr_dist_horiz = np.hypot(dx, dy)
        curr_dist_vert = abs(dz)
        
        # === FONCTION DE RÉCOMPENSE AMÉLIORÉE (REWARD SHAPING) ===
        reward = 0.0
        terminated = False
        
        # Seuil de collision réaliste et assoupli en espace normalisé
        if curr_dist_horiz < 0.15 and curr_dist_vert < 0.15:
            reward -= 50.0
            terminated = True  # Collision critique
        else:
            # Récompense basée sur le progrès : est-ce que l'action augmente la distance ?
            dist_delta = curr_dist_horiz - self.prev_dist
            reward += dist_delta * 5.0  # Encourage fortement l'éloignement
            
            # Bonus de confort si l'agent maintient une bonne distance
            if curr_dist_horiz > 1.5:
                reward += 0.5
        
        # Légère pénalité de temps pour inciter à résoudre le conflit rapidement
        reward -= 0.05
        
        self.prev_dist = curr_dist_horiz
        self.current_step += 1
        truncated = self.current_step >= self.max_steps
        
        return self.state, reward, terminated, truncated, {}

In [13]:
# Instanciation de l'environnement avec notre tenseur d'états
env = ATCEnv(initial_states_tensor)

# Réinitialisation pour démarrer un nouvel épisode
obs, info = env.reset()
print(f"--- DÉBUT DE L'ÉPISODE ---")
print(f"État initial (observation normalisée) : {obs}\n")

total_reward = 0.0
done = False
step_count = 0

# Boucle de simulation sur un épisode complet
while not done:
    # L'agent choisit une action totalement aléatoire (entre 0 et 4)
    action = env.action_space.sample()
    
    # Passage à l'état suivant via le simulateur
    next_obs, reward, terminated, truncated, info = env.step(action)
    
    total_reward += reward
    step_count += 1
    done = terminated or truncated
    
    print(f"Pas {step_count:02d} | Action: {action} | Reward: {reward:+.2f} | Collision/Arrêt: {terminated} | Limite temps: {truncated}")

print(f"\n--- FIN DE L'ÉPISODE ---")
print(f"Nombre total de pas : {step_count}")
print(f"Récompense cumulée de l'épisode : {total_reward:.2f}")

--- DÉBUT DE L'ÉPISODE ---
État initial (observation normalisée) : [ 0.30408928 -1.4749663   0.5302079   0.39269206  1.7139806   0.21434727]

Pas 01 | Action: 1 | Reward: -1.39 | Collision/Arrêt: False | Limite temps: False
Pas 02 | Action: 4 | Reward: -1.30 | Collision/Arrêt: False | Limite temps: False
Pas 03 | Action: 2 | Reward: -1.08 | Collision/Arrêt: False | Limite temps: False
Pas 04 | Action: 1 | Reward: -0.74 | Collision/Arrêt: False | Limite temps: False
Pas 05 | Action: 2 | Reward: -0.11 | Collision/Arrêt: False | Limite temps: False
Pas 06 | Action: 3 | Reward: +0.52 | Collision/Arrêt: False | Limite temps: False
Pas 07 | Action: 3 | Reward: +0.92 | Collision/Arrêt: False | Limite temps: False
Pas 08 | Action: 2 | Reward: +1.10 | Collision/Arrêt: False | Limite temps: False
Pas 09 | Action: 3 | Reward: +1.20 | Collision/Arrêt: False | Limite temps: False
Pas 10 | Action: 2 | Reward: +1.72 | Collision/Arrêt: False | Limite temps: False
Pas 11 | Action: 1 | Reward: +1.79 | C